# Project 2: Text Similarity & Duplicate Detection (G05)

## Notebook 3 (Approach 3): Fine-Tuning BERT (Cross-Encoder)

**Why this approach?**
* a Cross-Encoder feeds both questions into the model simultaneously.
* This allows the attention mechanism to directly compare every word in both questions, calculating the highest possible accuracy.

**Overview**
* Load the Quora Question Pairs dataset and initialize a HuggingFace tokenizer (`bert-base-uncased`).
* Convert the data into a HuggingFace `Dataset` and tokenize both questions together, separated by a `[SEP]` token.
* Load the pretrained BERT model with a binary sequence classification head (`num_labels=2`).
* Fine-tune the model using the HuggingFace `Trainer` API on a GPU to manage the heavy computation.
* Evaluate the final cross-encoder model's accuracy on the unseen test set.

Step 1: Setup HuggingFace Environment

Installation:

In [1]:
!pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


Libraries:

In [2]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
import os

Step 2: Data Preparation & Tokenization

In [4]:
import os
import pandas as pd

print("1. Loading dataset from local CSV file...")

data_path = "/content/sample_data/questions.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Could not find {data_path!r} in the current folder. "
        "Put your dataset CSV next to this notebook or change `data_path`."
    )

df = pd.read_csv(data_path)

# Keep only the columns we actually need for the model
required_cols = ["question1", "question2", "is_duplicate"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}. Found: {list(df.columns)[:20]}...")

# Drop nulls to prevent the tokenizer error
df = df[required_cols].dropna()

# SAMPLE THE DATA so computer survives the BERT fine-tuning
df = df.sample(n=10000, random_state=42)

print(f"-> Dataset loaded successfully with {df.shape[0]} rows!\n")

1. Loading dataset from local CSV file...
-> Dataset loaded successfully with 10000 rows!



In [5]:
# Force columns to string to keep the tokenizer happy
df['question1'] = df['question1'].astype(str)
df['question2'] = df['question2'].astype(str)

# Load tokenizer (using a lightweight BERT for faster training)
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Convert pandas dataframe to HuggingFace Dataset
hf_dataset = Dataset.from_pandas(df[['question1', 'question2', 'is_duplicate']])
hf_dataset = hf_dataset.rename_column("is_duplicate", "label")

# Split dataset
hf_dataset = hf_dataset.train_test_split(test_size=0.2)

# Tokenization function for pairs of text
def tokenize_function(example):
    return tokenizer(example["question1"], example["question2"], truncation=True, padding="max_length", max_length=128)

# Apply tokenization
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Step 3: Model Setup and Metrics

In [6]:
# Load pre-trained model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Define evaluation metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step 4: Training via Trainer API

In [7]:
# Load pre-trained model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Define evaluation metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting training... This may take a while!")
# Start fine-tuning
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training... This may take a while!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.464084,0.401878,0.811500
2,0.288592,0.417679,0.818000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.3763382263183594, metrics={'train_runtime': 448.4603, 'train_samples_per_second': 35.678, 'train_steps_per_second': 2.23, 'total_flos': 1052444221440000.0, 'train_loss': 0.3763382263183594, 'epoch': 2.0})

Step 5: Final Evaluation

In [10]:
# Evaluate on the test set
results = trainer.evaluate()
print("Final Evaluation Results:")
for key, value in results.items():
    # Clean up the key names (e.g., 'eval_loss' becomes 'Eval Loss')
    clean_key = key.replace('_', ' ').title()

    # Round numbers to 4 decimal places for a cleaner look
    if isinstance(value, float):
        print(f"{clean_key:<25}: {value:.4f}")
    else:
        print(f"{clean_key:<25}: {value}")

Final Evaluation Results:
Eval Loss                : 0.4177
Eval Accuracy            : 0.8180
Eval Runtime             : 19.7891
Eval Samples Per Second  : 101.0660
Eval Steps Per Second    : 6.3170
Epoch                    : 2.0000
